In [65]:
import json
import boto3
import time
from botocore.exceptions import ClientError

In [66]:
# Use this code snippet in your app.
# If you need more information about configurations
# or implementing the sample code, visit the AWS docs:
# https://aws.amazon.com/developer/language/python/

import boto3
from botocore.exceptions import ClientError


def get_secret():

    secret_name = "ope-serverless-redshift"
    region_name = "eu-west-2"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    secret = get_secret_value_response['SecretString']
    return json.loads(secret)
secret_dict = get_secret()


In [67]:

# configurations 
REGION_NAME = "eu-west-2"                 
REDSHIFT_DATABASE = "bmoni_assessment"                  
REDSHIFT_WORKGROUP_NAME = "default-workgroup" 


S3_BUCKET = "od-bmoni-assessment"
S3_KEY = "assessment_data/application_data - application_data.csv"
IAM_ROLE_ARN = secret_dict['role_arn']
accesskey_id = secret_dict['accesskey_id']
access_key = secret_dict['access_key']

secrets_client = boto3.client("secretsmanager", region_name=REGION_NAME)
redshift_data = boto3.client("redshift-data", region_name=REGION_NAME)


In [68]:
def execute_redshift_sql(sql):
    """Run SQL command on Redshift using Data API and wait for completion."""
    # Retrieve secret
    creds = get_secret()

    redshift_data = boto3.client('redshift-data', region_name='eu-west-2')

    # Build execution parameters
    params = {
        "Database": secret_dict['dbname'],  
        "SecretArn": secret_dict['secret_arn'],  
    }

    # option to connect to either Serverless or Redshift cluster
    if 'REDSHIFT_WORKGROUP_NAME' in globals() and REDSHIFT_WORKGROUP_NAME:
        params["WorkgroupName"] = REDSHIFT_WORKGROUP_NAME
    elif 'REDSHIFT_CLUSTER_ID' in globals() and REDSHIFT_CLUSTER_ID:
        params["ClusterIdentifier"] = REDSHIFT_CLUSTER_ID
    else:
        raise ValueError("Either REDSHIFT_WORKGROUP_NAME or REDSHIFT_CLUSTER_ID must be set")

    params["Sql"] = sql

    # Execute SQL
    response = redshift_data.execute_statement(**params)
    statement_id = response["Id"]

    # execution status logging
    while True:
        desc = redshift_data.describe_statement(Id=statement_id)
        status = desc["Status"]
        if status in ("FAILED", "FINISHED", "ABORTED"):
            break
        time.sleep(2)

    if status == "FAILED":
        raise Exception(f"Query failed: {desc.get('Error')}")

    return desc



In [69]:
# execute copy command to migrate data from S3 to Redshift
def main():
    print("Starting data migration from S3 to Redshift...")

    copy_sql = f"""
        COPY staging.application_data
        FROM 's3://{S3_BUCKET}/{S3_KEY}'
        ACCESS_KEY_ID '{accesskey_id}'
        SECRET_ACCESS_KEY '{access_key}'
        FORMAT AS CSV
        IGNOREHEADER 1
        DELIMITER ','
        TIMEFORMAT 'auto'
        ACCEPTINVCHARS;
    """

    print("Executing COPY command...")
    result = execute_redshift_sql(copy_sql)
    print("COPY completed with status:", result["Status"])


In [70]:
if __name__ == "__main__":
    main()

Starting data migration from S3 to Redshift...
Executing COPY command...
COPY completed with status: FINISHED
